# Statistical Analysis & Visualization — Exploratory Data Analysis (EDA)
**Internship task | Python · Jupyter Notebook · Pandas · Matplotlib · Seaborn**

## Objective
Perform exploratory data analysis on the UCI Online Retail dataset. Examine distributions, compare trends, calculate correlations, create heatmaps and distribution plots, and annotate important findings.

## Dataset
Download the **Online Retail** dataset from the official UCI Machine Learning Repository:
https://archive.ics.uci.edu/dataset/352/online+retail

Place `Online Retail.xlsx` beside this notebook, or change `FILE_PATH` in Section 2.

## Important interpretation notes
- Correlation describes association, not causation.
- Retail data can include returns/credits (negative quantities) and unusual unit prices. Inspect these rather than deleting them automatically.
- Revenue here is calculated as `Quantity × UnitPrice`. Negative values can therefore reflect returns/credits.
- Statistical tests and p-values should be interpreted with assumptions and sample size in mind. Large datasets can make tiny differences statistically significant.

## 1. Install required packages

In [ ]:
%pip install pandas numpy matplotlib seaborn scipy openpyxl

## 2. Import libraries and load the dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

FILE_PATH = Path("Online Retail.xlsx")
if not FILE_PATH.exists():
    raise FileNotFoundError(
        "Could not find Online Retail.xlsx. Put it beside this notebook "
        "or update FILE_PATH to the correct file location."
    )

df = pd.read_excel(FILE_PATH)
df.columns = df.columns.astype(str).str.strip()

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 3. Initial data exploration

In [ ]:
display(df.info())
display(df.describe(include="all").T)

missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)
display(missing)

print("Exact duplicate rows:", int(df.duplicated().sum()))

## 4. Prepare analysis columns

In [ ]:
# Work on a copy; retain the original raw data in df if you want to compare.
eda = df.copy()

for col in ["Quantity", "UnitPrice"]:
    eda[col] = pd.to_numeric(eda[col], errors="coerce")

eda["InvoiceDate"] = pd.to_datetime(eda["InvoiceDate"], errors="coerce")
eda["Revenue"] = eda["Quantity"] * eda["UnitPrice"]
eda["YearMonth"] = eda["InvoiceDate"].dt.to_period("M").astype("string")
eda["Hour"] = eda["InvoiceDate"].dt.hour
eda["IsReturnOrCredit"] = eda["Quantity"] < 0

print("Prepared analysis columns.")
display(eda[["InvoiceDate", "Quantity", "UnitPrice", "Revenue", "YearMonth", "Hour"]].head())

## 5. Univariate analysis — distributions
Histograms help us see the shape and spread of numeric variables. Retail revenue and quantities can be highly skewed, so use sensible axis limits for visualization and explain any filtering.

In [ ]:
numeric_cols = [c for c in ["Quantity", "UnitPrice", "Revenue"] if c in eda.columns]

for col in numeric_cols:
    values = eda[col].replace([np.inf, -np.inf], np.nan).dropna()
    # Clip only the plot display to the 1st–99th percentile to make the main body visible.
    # The underlying data are not changed.
    if len(values) > 0:
        lo, hi = values.quantile([0.01, 0.99])
        plot_values = values[values.between(lo, hi)]
        plt.figure(figsize=(8, 4))
        sns.histplot(plot_values, bins=40, kde=True)
        plt.title(f"Distribution of {col} (displayed between 1st and 99th percentiles)")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()

## 6. Boxplots — identify potential outliers

In [ ]:
for col in ["Quantity", "UnitPrice", "Revenue"]:
    values = eda[col].replace([np.inf, -np.inf], np.nan).dropna()
    if len(values):
        lo, hi = values.quantile([0.01, 0.99])
        plt.figure(figsize=(8, 2.8))
        sns.boxplot(x=values[values.between(lo, hi)])
        plt.title(f"{col}: boxplot (1st–99th percentile display)")
        plt.tight_layout()
        plt.show()

## 7. Correlation analysis
Pearson correlation measures linear association between numeric variables. Because revenue is computed from quantity and unit price, its correlation with those variables is partly mathematical and should not be interpreted as an independent discovery.

In [ ]:
corr_cols = [c for c in ["Quantity", "UnitPrice", "Revenue", "Hour"] if c in eda.columns]
corr_data = eda[corr_cols].replace([np.inf, -np.inf], np.nan).dropna()
corr_matrix = corr_data.corr(method="pearson")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Pearson Correlation Heatmap")
plt.tight_layout()
plt.show()

display(corr_matrix)

## 8. Compare trends over time

In [ ]:
monthly = (
    eda.dropna(subset=["InvoiceDate"])
       .groupby(eda["InvoiceDate"].dt.to_period("M"))
       .agg(Total_Revenue=("Revenue", "sum"),
            Invoice_Count=("InvoiceNo", "nunique"),
            Total_Quantity=("Quantity", "sum"))
       .reset_index(names="YearMonth")
)
monthly["YearMonth"] = monthly["YearMonth"].astype(str)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly, x="YearMonth", y="Total_Revenue", marker="o")
plt.title("Monthly Revenue Trend (includes negative return/credit lines)")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

display(monthly.head())

## 9. Compare categories — countries and transaction types

In [ ]:
if "Country" in eda.columns:
    country_revenue = (
        eda.groupby("Country", dropna=False)["Revenue"]
           .sum()
           .sort_values(ascending=False)
           .head(10)
           .reset_index()
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=country_revenue, x="Revenue", y="Country")
    plt.title("Top 10 Countries by Net Revenue")
    plt.xlabel("Net Revenue")
    plt.ylabel("Country")
    plt.tight_layout()
    plt.show()
    display(country_revenue)

eda["Transaction_Type"] = np.select(
    [eda["Quantity"] < 0, eda["Quantity"] == 0, eda["Quantity"] > 0],
    ["Return/Credit", "Zero quantity", "Positive quantity"],
    default="Unknown"
)

plt.figure(figsize=(7, 4))
sns.countplot(data=eda, x="Transaction_Type",
              order=eda["Transaction_Type"].value_counts().index)
plt.title("Transaction Line Counts by Quantity Sign")
plt.xlabel("Transaction type")
plt.ylabel("Number of lines")
plt.tight_layout()
plt.show()

## 10. Annotate key chart findings
Annotations should be based on the actual plot results. This example labels the largest country by net revenue, rather than hardcoding a country name or result.

In [ ]:
if "Country" in eda.columns:
    country_revenue_all = (
        eda.groupby("Country", dropna=False)["Revenue"]
           .sum()
           .sort_values(ascending=False)
           .head(10)
           .reset_index()
    )
    if not country_revenue_all.empty:
        top_row = country_revenue_all.iloc[0]
        plt.figure(figsize=(10, 5))
        ax = sns.barplot(data=country_revenue_all, x="Revenue", y="Country")
        ax.annotate(
            f"Highest in this chart: {top_row['Country']}\n{top_row['Revenue']:,.2f}",
            xy=(top_row["Revenue"], 0),
            xytext=(0.55, 0.88), textcoords="axes fraction",
            arrowprops={"arrowstyle": "->"},
            ha="left", va="center"
        )
        ax.set_title("Top 10 Countries by Net Revenue — with annotation")
        ax.set_xlabel("Net Revenue")
        ax.set_ylabel("Country")
        plt.tight_layout()
        plt.show()

## 11. Statistical significance example
We can compare line-level revenue for two groups, positive-quantity lines and return/credit lines, using the Mann–Whitney U test. This is an illustrative comparison, not a causal analysis. These groups may have very different distributions and sample sizes.

Because the dataset can be large, the test may be statistically significant even when the practical difference is not important. Report group medians and sample sizes alongside the p-value.

In [ ]:
positive = eda.loc[eda["Quantity"] > 0, "Revenue"].replace([np.inf, -np.inf], np.nan).dropna()
returns = eda.loc[eda["Quantity"] < 0, "Revenue"].replace([np.inf, -np.inf], np.nan).dropna()

print("Positive-quantity lines:", len(positive))
print("Return/credit lines:", len(returns))
print("Positive-quantity median revenue:", positive.median() if len(positive) else np.nan)
print("Return/credit median revenue:", returns.median() if len(returns) else np.nan)

if len(positive) > 0 and len(returns) > 0:
    test = stats.mannwhitneyu(positive, returns, alternative="two-sided")
    print("Mann–Whitney U statistic:", test.statistic)
    print("p-value:", test.pvalue)
    alpha = 0.05
    if test.pvalue < alpha:
        print("Result: statistically significant difference at alpha = 0.05.")
    else:
        print("Result: insufficient evidence of a difference at alpha = 0.05.")
else:
    print("Not enough observations in both groups to run the test.")

## 12. Optional: export EDA summaries

In [ ]:
output_dir = Path("eda_output")
output_dir.mkdir(exist_ok=True)

missing.to_csv(output_dir / "missing_value_report.csv")
corr_matrix.to_csv(output_dir / "correlation_matrix.csv")
monthly.to_csv(output_dir / "monthly_trend.csv", index=False)

if "country_revenue" in globals():
    country_revenue.to_csv(output_dir / "top_country_revenue.csv", index=False)

print("EDA summary files saved to:", output_dir.resolve())

## 13. Findings template — complete after running the notebook
Do not invent findings. Use the actual charts and numerical outputs to complete the following:

1. **Distribution:** Describe whether Quantity, UnitPrice, and Revenue are skewed and mention visible extreme values.
2. **Correlation:** Report one or two correlation values from the heatmap and explain that correlation does not establish causation.
3. **Trend:** Identify the month with the highest and lowest net revenue in the monthly summary.
4. **Category comparison:** Name the countries shown at the top of the net-revenue chart.
5. **Statistical test:** Report group sample sizes, medians, test statistic, and p-value; explain the 0.05 threshold and practical limitations.

## Conclusion
This notebook demonstrates exploratory data analysis with Pandas, Matplotlib, Seaborn, Pearson correlation, distribution plots, boxplots, time trends, chart annotations, and an example non-parametric significance test.